In [17]:
from ures.string import format_memory
from experiments_llm.evaluation import LLMEvaluator
import GPUtil
import torch
import time
import os

In [18]:
if torch.cuda.device_count() > 1:
    os.environ["NCCL_P2P_DISABLE"] = "1"
    os.environ["NCCL_IB_DISABLE"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":0:0"
torch.backends.cuda.cufft_plan_cache.max_size = 1


In [19]:
models = [
    "EleutherAI/gpt-neo-125M",
    "facebook/opt-125m",
    "facebook/opt-350m",
    "cerebras/Cerebras-GPT-111M",
    "microsoft/deberta-base",
    "T5-small",
    "t5-base",
    "distilbert/distilgpt2",
    "openai-community/gpt2",
    # "bigscience/bloom-560m",
]
model = models[-1]
device_id = 0
batch = 5
target_iteration = 2
optimiser = "AdamW"

In [20]:
evaluator = LLMEvaluator(
    model_name=model,
    optimizer=optimiser,
    batch_size=batch,
    gpu_id=device_id,
    iterations=target_iteration,
)

The first run on CPU

In [21]:
c_config = evaluator.train_on_cpu()
print(f"The result is located at {c_config.result_dir}")

The result is located at /home/glaswegian/DL-Estimator/bigscience-bloom-560m_AdamW_5_7f79/CPU/results


Start estimating Max GPU memory

In [22]:
c_est, c_result = evaluator.get_estimate_max_gpu(c_config)
est_seg = max(c_est._trace.max_segment_changes)
est_oom  = c_est.oom
print(f"The result:{c_result}, the max segment changes:{est_seg}")


/home/glaswegian/DL-Estimator/bigscience-bloom-560m_AdamW_5_7f79/CPU/results
The result:{'OOM': False, 'Max GPU Memory': 16621523435.52, 'memory': {'tensor': 11060439552, 'segment': 12501123072}}, the max segment changes:12501123072


In [23]:
c_est.plot_memory_change()


Start first training on GPU

In [24]:
try:
    g_config = evaluator.train_on_gpu()
    g_nvml = evaluator.get_truth_ground_max_gpu(g_config)
except Exception as e:
    print(e)
    real_oom = True
    ground = None
else:
    real_oom = False
    ground = max(g_nvml[str(evaluator.g_id)])

if real_oom:
    print(f"OOM occurred")
else:
    print(f"The real memory usage is {format_memory(ground)}")

Set GPU Memory Fraction: 100.0%, limit: 0 bytes
/home/glaswegian/DL-Estimator/bigscience-bloom-560m_AdamW_5_7f79/GPU-447f/results/host_monitor/host_metrics-1744413589.json
The real memory usage is 12.65 GB


In [25]:
torch.cuda.empty_cache()
time.sleep(3)
framework_memory_usage = GPUtil.getGPUs()[device_id].memoryUsed * 1024**2
total_memory_used = est_seg + framework_memory_usage
print(f"[{real_oom}]Ground Truth Memory: {format_memory(ground)}")
print(f"Framework Memory Usage: {format_memory(framework_memory_usage)}")
print(f"Total Memory Will Be Set: {format_memory(total_memory_used)}")


[False]Ground Truth Memory: 12.65 GB
Framework Memory Usage: 349.00 MB
Total Memory Will Be Set: 11.98 GB


Record 1st verification

In [26]:
all_info = evaluator.all_result
all_info[evaluator.tool_name] = {
    "ground": ground,
    "framework": framework_memory_usage,
    "memory": est_seg,
    "oom": est_oom,
    "real_oom": real_oom,
    "error": None if real_oom else abs(est_seg - ground) / ground,
    "correct_estimation": est_oom == real_oom,
    "2nd verification": {}
}

print(f"The 1st round verification's error: {all_info[evaluator.tool_name]['error']}")

The 1st round verification's error: 0.07938223938223939


Start verifying of estimated memory

In [27]:
if real_oom is False and all_info[evaluator.tool_name]["correct_estimation"]:
    try:
        print(f"The max memory set to {format_memory(est_seg)}(Est) + {format_memory(framework_memory_usage)}(Framework)")
        g_config_2nd = evaluator.train_on_gpu(gpu_memory_in_bytes=total_memory_used)
        g_nvml_2nd = evaluator.get_truth_ground_max_gpu(g_config_2nd)
    except Exception as e:
        print(f"2nd verification run error: {e}")
        real_oom_2nd = True
        ground_2nd = None
    else:
        real_oom_2nd = False
        ground_2nd = max(g_nvml_2nd[str(evaluator.g_id)])

    torch.cuda.empty_cache()
    time.sleep(3)
    tool_data = all_info[evaluator.tool_name]
    tool_data["2nd verification"] = {
        "oom": real_oom_2nd,
        "ground": ground_2nd,
        "error": None if real_oom_2nd else abs(est_seg - ground_2nd) / ground_2nd,
    }

    print(f"The second verification's error: {tool_data["2nd verification"]['error']}")
else:
    print(f"Due to both real OOM and estimated OOM are `True`, the second verification is skipped.")



The max memory set to 11.64 GB(Est) + 349.00 MB(Framework)
Set GPU Memory Fraction: 77.0%, limit: 11.98 GB


Exception occurred during training.
CUDA out of memory. Tried to allocate 608.00 MiB. GPU 0 has a total capacity of 15.48 GiB of which 3.67 GiB is free. Including non-PyTorch memory, this process has 11.79 GiB memory in use. 11.92 GiB allowed; Of the allocated memory 9.32 GiB is allocated by PyTorch, and 2.15 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


2nd verification run error: Training failed: CUDA out of memory. Tried to allocate 608.00 MiB. GPU 0 has a total capacity of 15.48 GiB of which 3.67 GiB is free. Including non-PyTorch memory, this process has 11.79 GiB memory in use. 11.92 GiB allowed; Of the allocated memory 9.32 GiB is allocated by PyTorch, and 2.15 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
The second verification's error: None


In [28]:
import json
all_info["config"] = evaluator.config.model_dump()
report_dir = evaluator.config.base_dir
with open(report_dir.joinpath(f"evaluation_result.json"), "w") as f:
    json.dump(all_info, f, indent=4)